In [52]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import Counter
import matplotlib.pyplot as plt
import re
from scipy.stats import spearmanr

In [3]:
from pandas import DataFrame

DATA_DIR = Path("./patienten-visiten")
csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))
all_visits = {}

for file in csv_files:
    df = pd.read_csv(file, sep=";")

    filename = file.stem
    visit = filename.split("_")[1]

    df["visit"] = visit

    all_visits[visit] = df

C:\Users\veron\AppData\Local\Temp\ipykernel_2780\252853293.py:8: DtypeWarning: Columns (106,107,228,230,243,245,247,249,251,252,260,262,264,266,268,269,277,279,281,283,285,286,294,300,302,303,311,313,371,373,374,383,385,386,395,396,397,398,399,407,408,409,410,411,416,418,419,420,421,422,423,428,430,431,432,433,435,440,441,442,443,444,445,446,447,448,450,451,452,453,454,459,460,462,463,464,465,466,470,472,474,475,476,477,484,486,487,488,489,490,496,498,499,501,502,508,510,511,513,730,735,737,755,756,761,762,763,777,779,781,782,788,791,799,801,803,805,808,814,817,825,827,829,1016,1024,1028,1033,1034,1040,1041,1042,1046,1048,1051,1052,1058,1059,1060,1064,1066,1069,1070,1076,1077,1078,1082,1084,1087,1094,1095,1096,1099,1100,1102,1105,1112,1113,1114,1115,1117,1118,1120,1123,1124,1130,1131,1132,1133,1134,1135,1136,1138,1141,1144,1145,1148,1149,1151,1152,1153,1156,1161,1162,1163,1166,1167,1169,1170,1171,1174,1177,1178,1179,1180,1181,1184,1185,1187,1188,1189,1195,1197,1198,1199,1202,1203,1204,

In [4]:
top_meds_per_visit = {}

for visit, df in all_visits.items():
    MED_COLUMN = f"acute_name_a1_K{visit[1:]}"
    if MED_COLUMN not in df.columns:
        print(f"{visit}: Spalte {MED_COLUMN} nicht gefunden")
        continue

    meds = (
        df[MED_COLUMN]
        .dropna()
        .astype(str)
        .str.strip()
    )

    meds = meds[(meds != "") & (meds != "0")]

    top_meds = meds.value_counts().head(20)

    top_meds_per_visit[visit] = top_meds

Vlast: Spalte acute_name_a1_Klast nicht gefunden


In [5]:
dfs = []
PATIENT_ID_COLUMN = "id_patient"

for visit, df in all_visits.items():

    med_col = f"acute_name_a1_K{visit[1:]}"

    if med_col not in df.columns:
        print(f"{visit}: Spalte {med_col} fehlt")
        continue

    temp = df[[PATIENT_ID_COLUMN, med_col]].copy()
    temp["visit"] = visit

    temp = temp.rename(columns={med_col: "medication"})

    dfs.append(temp)

combined_df = pd.concat(dfs, ignore_index=True)

print("\nGesamte Zeilen:", len(combined_df))

Vlast: Spalte acute_name_a1_Klast fehlt

Gesamte Zeilen: 17866


In [6]:
PATIENT_ID_COLUMN = "id_patient"


relevant = combined_df.dropna(subset=[PATIENT_ID_COLUMN])

relevant["medication"] = (
    relevant["medication"]
    .fillna("")
    .astype(str)
    .str.strip()
)

relevant = relevant[(relevant["medication"] != "") & (relevant["medication"] != "0")]
visit_order = sorted(
    [v for v in all_visits.keys() if v[1:].isdigit()],
    key=lambda v: int(v[1:])
)

print("\nVisit Reihenfolge:")
print(visit_order)


Visit Reihenfolge:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23']


In [7]:
patient_med_matrix = (
    relevant
    .assign(value=1)
    .pivot_table(
        index="id_patient",
        columns="medication",
        values="value",
        aggfunc="max",
        fill_value=0
    )
)

In [8]:
corr = patient_med_matrix.corr()

In [9]:
target_med = "Tilidin"

patient_med_matrix.corrwith(patient_med_matrix[target_med]).sort_values(ascending=False)

medication
Tilidin                                      1.000000
Zaldiar® 37,5 mg/325 mg                      0.266978
Zolmitriptan                                 0.023534
Thomapyrin® intensiv                         0.010199
Metamizol                                    0.004963
Eletriptan                                   0.000968
Sumatriptan                                  0.000223
Dolopyrin®                                  -0.000610
Vivimed® mit Coffein gegen Kopfschmerzen    -0.000610
Ergotamintartrat                            -0.000610
Tramabian® 37,5 mg/300 mg                   -0.000610
Rimegepant                                  -0.000610
Migraeflux® MCP                             -0.000610
Tramabian® 75 mg/650 mg                     -0.000610
Doppel Spalt®                               -0.000610
Ondansetron                                 -0.000610
Togal® Kopfschmerz-Brause + Vitamin C       -0.000610
Talvosilen® forte                           -0.000610
Voltaren® plus   

In [10]:
DATA_DIR = Path("./patienten-visiten")
csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))

In [11]:
dfs = [pd.read_csv(f, sep=";") for f in csv_files]
data = pd.concat(dfs, ignore_index=True)


C:\Users\veron\AppData\Local\Temp\ipykernel_2780\2915578457.py:1: DtypeWarning: Columns (106,107,228,230,243,245,247,249,251,252,260,262,264,266,268,269,277,279,281,283,285,286,294,300,302,303,311,313,371,373,374,383,385,386,395,396,397,398,399,407,408,409,410,411,416,418,419,420,421,422,423,428,430,431,432,433,435,440,441,442,443,444,445,446,447,448,450,451,452,453,454,459,460,462,463,464,465,466,470,472,474,475,476,477,484,486,487,488,489,490,496,498,499,501,502,508,510,511,513,730,735,737,755,756,761,762,763,777,779,781,782,788,791,799,801,803,805,808,814,817,825,827,829,1016,1024,1028,1033,1034,1040,1041,1042,1046,1048,1051,1052,1058,1059,1060,1064,1066,1069,1070,1076,1077,1078,1082,1084,1087,1094,1095,1096,1099,1100,1102,1105,1112,1113,1114,1115,1117,1118,1120,1123,1124,1130,1131,1132,1133,1134,1135,1136,1138,1141,1144,1145,1148,1149,1151,1152,1153,1156,1161,1162,1163,1166,1167,1169,1170,1171,1174,1177,1178,1179,1180,1181,1184,1185,1187,1188,1189,1195,1197,1198,1199,1202,1203,1204

In [12]:
med_cols = [c for c in data.columns if c.startswith("acute_name_a")]

In [13]:
score_bases = [
    "midas_result",
    "dass_depression",
    "dass_fear",
    "dass_stress",
    "vr12_msc",
    "vr12_psc",
    "gvas_result",
    "pgic_result",
    "chiq_result"
]

In [14]:
def analyze_score(score_base):

    # alle K-Versionen finden
    score_cols = [c for c in data.columns if c.startswith(score_base)]

    if len(score_cols) == 0:
        print(f"Keine Spalten gefunden für {score_base}")
        return None

    all_results = []

    for score_col in score_cols:

        for med_col in med_cols:

            tmp = data[[med_col, score_col]].copy()
            tmp.columns = ["medication", "score"]

            tmp = tmp.dropna()

            tmp = tmp[
                (tmp["medication"] != "") &
                (tmp["medication"] != "0")
            ]

            tmp["score"] = pd.to_numeric(tmp["score"], errors="coerce")
            tmp = tmp.dropna(subset=["score"])

            if len(tmp) == 0:
                continue

            agg = tmp.groupby("medication")["score"].agg(
                count="count",
                mean="mean",
                median="median"
            )

            all_results.append(agg)

    if len(all_results) == 0:
        return None

    return (
        pd.concat(all_results)
        .groupby(level=0)
        .mean()
        .sort_values("mean", ascending=False)
    )

In [15]:
results = {}

for s in score_bases:
    print(f"Processing: {s}")
    res = analyze_score(s)
    if res is not None:
        results[s] = res

Processing: midas_result
Processing: dass_depression
Processing: dass_fear
Processing: dass_stress
Processing: vr12_msc
Processing: vr12_psc
Processing: gvas_result
Processing: pgic_result
Processing: chiq_result


In [16]:
results["midas_result"].head(20)

,count,mean,median
medication,,,
"Tramabian® 37,5 mg/300 mg",1.000000,160.000000,160.000000
Tilidin,2.229167,158.812721,157.041667
"Zaldiar® 37,5 mg/325 mg",1.000000,140.000000,140.000000
Rimegepant,1.000000,125.000000,125.000000
Tramadol,2.045455,121.278409,116.045455
Azur® compositum Tabletten,1.000000,93.875000,93.875000
Gelonida®,1.227273,89.787879,89.909091
Ondansetron,1.448276,86.888506,86.258621
Sauerstoff,3.625000,82.823691,76.734375


In [17]:
results["dass_depression"].head(20)

,count,mean,median
medication,,,
Titralgan® gegen Schmerzen,1.000000,13.000000,13.000000
Phenazon,1.250000,10.125000,10.125000
Indometacin,3.235294,7.673174,7.352941
Tramabian® 75 mg/650 mg,1.000000,7.666667,7.666667
Paracetamol/Ibuprofen Acino,3.818182,7.089394,6.954545
Tramadol,2.045455,6.931818,6.727273
Sauerstoff,3.625000,6.920641,6.734375
Tilidin,2.229167,6.790215,6.864583
Diclofenac,3.862069,6.465893,6.068966


In [18]:
results["dass_fear"].head(20)

,count,mean,median
medication,,,
Vivimed® mit Coffein gegen Kopfschmerzen,1.000000,7.000000,7.000000
Paracetamol/Ibuprofen Acino,3.818182,6.741667,6.681818
Migraeflux® MCP,1.111111,6.611111,6.611111
Phenazon,1.250000,6.125000,6.125000
Tilidin,2.229167,6.082449,6.041667
Gelonida®,1.227273,5.924242,5.500000
Tramadol,2.045455,5.888258,5.795455
Lidocain nasal,2.000000,5.500000,5.500000
Spalt® Schmerztabletten 300 mg/300 mg,1.600000,4.600000,4.600000


In [19]:
results["dass_stress"].head(20)

,count,mean,median
medication,,,
Titralgan® gegen Schmerzen,1.000000,17.000000,17.000000
Gelonida®,1.227273,12.113636,12.159091
Octadon®,1.250000,11.625000,11.625000
Phenazon,1.250000,10.875000,10.875000
Tilidin,2.229167,10.671843,10.666667
Tramabian® 75 mg/650 mg,1.000000,10.555556,10.555556
Migraeflux® MCP,1.111111,10.277778,10.277778
Paracetamol AL comp.,1.521739,10.137681,10.239130
"Tramabian® 37,5 mg/300 mg",1.000000,9.400000,9.400000


In [20]:
results["gvas_result"].head(20)

,count,mean,median
medication,,,
Para-Caf®,1.000000,100.000000,100.000000
Dolopyrin®,1.000000,100.000000,100.000000
Doppel Spalt®,1.000000,96.333333,96.333333
Suvexx® (Sumatriptan/Naproxen 85mg/500mg),1.000000,80.800000,80.800000
Voltaren® plus,1.000000,71.333333,71.333333
Spalt® Schmerztabletten 300 mg/300 mg,1.600000,67.900000,67.900000
Togal® Kopfschmerz-Brause + Vitamin C,1.000000,67.777778,67.777778
Thomapyrin® Emra,2.333333,67.222222,67.666667
Gelonida®,1.227273,57.219697,56.113636


In [21]:
results["pgic_result"].head(20)

,count,mean,median
medication,,,
Tilidin,1.461538,5.207265,5.166667
"Tramabian® 37,5 mg/300 mg",1.000000,5.000000,5.000000
Talvosilen® 500 mg/20 mg Tabletten,1.000000,5.000000,5.000000
Phenazon,1.000000,5.000000,5.000000
Aspirin® Coffein,1.000000,5.000000,5.000000
Azur® compositum Tabletten,1.000000,5.000000,5.000000
Tramadol,1.235294,4.735294,4.735294
Gelonida®,1.166667,4.666667,4.722222
Sauerstoff,2.107143,4.464711,4.375000


In [22]:
results["chiq_result"].head(20)

,count,mean,median
medication,,,
Indometacin,1.000000,31.000000,31.000000
Dimenhydrinat,1.000000,28.000000,28.000000
Thomapyrin® intensiv,2.200000,27.516667,27.500000
Tramadol,1.000000,27.000000,27.000000
Lasmiditan,1.000000,27.000000,27.000000
Acetylsalicylsäure,1.000000,22.500000,22.500000
Neuralgin® Schmerztabletten,1.000000,19.000000,19.000000
Sauerstoff,3.129032,17.485496,17.516129
Zolmitriptan,3.500000,16.192923,16.285714


In [46]:
DATA_DIR = Path("./patienten-visiten")
csv_files = sorted(DATA_DIR.glob("Patients_V*.csv"))

dfs = [pd.read_csv(f, sep=";") for f in csv_files]
data = pd.concat(dfs, ignore_index=True)

C:\Users\veron\AppData\Local\Temp\ipykernel_2780\2619675657.py:4: DtypeWarning: Columns (106,107,228,230,243,245,247,249,251,252,260,262,264,266,268,269,277,279,281,283,285,286,294,300,302,303,311,313,371,373,374,383,385,386,395,396,397,398,399,407,408,409,410,411,416,418,419,420,421,422,423,428,430,431,432,433,435,440,441,442,443,444,445,446,447,448,450,451,452,453,454,459,460,462,463,464,465,466,470,472,474,475,476,477,484,486,487,488,489,490,496,498,499,501,502,508,510,511,513,730,735,737,755,756,761,762,763,777,779,781,782,788,791,799,801,803,805,808,814,817,825,827,829,1016,1024,1028,1033,1034,1040,1041,1042,1046,1048,1051,1052,1058,1059,1060,1064,1066,1069,1070,1076,1077,1078,1082,1084,1087,1094,1095,1096,1099,1100,1102,1105,1112,1113,1114,1115,1117,1118,1120,1123,1124,1130,1131,1132,1133,1134,1135,1136,1138,1141,1144,1145,1148,1149,1151,1152,1153,1156,1161,1162,1163,1166,1167,1169,1170,1171,1174,1177,1178,1179,1180,1181,1184,1185,1187,1188,1189,1195,1197,1198,1199,1202,1203,1204

In [ ]:
score_bases = [
    "midas_result",
    "dass_depression",
    "dass_fear",
    "dass_stress",
    "vr12_msc",
    "vr12_psc",
    "gvas_result",
    "pgic_result",
    "chiq_result"
]

score_cols = [c for c in data.columns if any(b in c for b in score_bases)]

score_long = data.melt(
    id_vars=["id_patient"],
    value_vars=score_cols,
    var_name="score_type_k",
    value_name="score"
)

# numeric cleanup
score_long["score"] = pd.to_numeric(score_long["score"], errors="coerce")

# extract k consistently as numeric string
score_long["k"] = score_long["score_type_k"].str.extract(r"K(\d+)")[0]

# clean score type
score_long["score_type"] = score_long["score_type_k"].str.replace(r"_K\d+", "", regex=True)

score_long = score_long.dropna(subset=["score", "k"])

In [ ]:
def extract_acute_table(df):

    rows = []

    pattern = re.compile(r"acute_name_a(\d+)_K(\d+)")

    for col in df.columns:

        m = pattern.search(col)
        if not m:
            continue

        a_idx = m.group(1)
        k_idx = m.group(2)

        name_col = f"acute_name_a{a_idx}_K{k_idx}"
        dose_col = f"dosage_a{a_idx}_K{k_idx}"

        if name_col not in df.columns or dose_col not in df.columns:
            continue

        tmp = df[["id_patient", name_col, dose_col]].copy()
        tmp.columns = ["id_patient", "medication", "dose"]

        tmp["slot"] = a_idx

        # IMPORTANT FIX: match score_long format (numeric string)
        tmp["k"] = k_idx

        rows.append(tmp)

    if not rows:
        return pd.DataFrame(columns=["id_patient", "medication", "dose", "slot", "k"])

    out = pd.concat(rows, ignore_index=True)

    out["medication"] = out["medication"].astype(str).str.strip()
    out["dose"] = pd.to_numeric(out["dose"], errors="coerce")
    out["k"] = out["k"].astype(str)

    out = out.dropna(subset=["medication", "dose"])

    return out


In [132]:
med_df = extract_acute_table(data)

In [139]:
tramabian_df = med_df[
    med_df["medication"].str.startswith("Tramabian", na=False)
].copy()

In [140]:
tramabian_df = tramabian_df.dropna(subset=["dose", "medication"])

In [141]:
merged = tramabian_df.merge(
    score_long,
    on=["id_patient", "k"],
    how="inner"
)

merged = merged.dropna(subset=["dose", "score"])

# remove invalid values
merged = merged[np.isfinite(merged["dose"])]
merged = merged[np.isfinite(merged["score"])]

In [143]:
print("Final dataset size:", merged.shape)

print("\nDose distribution:")
print(merged["dose"].describe())

print("\nK distribution:")
print(merged["k"].value_counts().head())

Final dataset size: (0, 8)

Dose distribution:
count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
Name: dose, dtype: float64

K distribution:
Series([], Name: count, dtype: int64)


In [138]:
results = []

MIN_N = 10

for score_type, df in merged.groupby("score_type"):

    if len(df) < MIN_N:
        continue

    corr, p = spearmanr(df["dose"], df["score"], nan_policy="omit")

    results.append({
        "score_type": score_type,
        "n": len(df),
        "spearman_corr": corr,
        "p_value": p
    })

results_df = pd.DataFrame(results)
results_df = results_df.sort_values("spearman_corr", ascending=False)

KeyError: 'spearman_corr'

In [144]:
print("\n=== Tramabian Dose vs Score Correlations ===\n")
display(results_df)


=== Tramabian Dose vs Score Correlations ===



""
